# ocrpipe di Kaggle / Google Colab

Notebook ini menjalankan **engine OCR berat** (Docling / PaddleOCR PP-StructureV3 / Marker) di GPU gratis, lalu mengunduh hasilnya untuk dipakai di sistem RAG skripsi.

**Kapan pakai ini daripada laptop:** ketika korpus punya banyak halaman hasil scan yang bertabel. GPU Kaggle (T4/P100, 16 GB) dan RAM ~29 GB jauh melampaui laptop (RTX 4050 6 GB / RAM 8 GB), dan RAM besar inilah yang mengatasi bottleneck sebenarnya.

**Cara pakai (Kaggle):**
1. Settings > Accelerator > **GPU T4 x2** (atau P100).
2. Settings > Internet > **On** (perlu untuk unduh model & paket).
3. Upload `ocrpipe` (folder) + PDF kamu sebagai *Dataset*, atau pakai sel upload di bawah.
4. Jalankan sel berurutan.

**Cara pakai (Colab):** Runtime > Change runtime type > **GPU**, lalu jalankan.

## 1. Deteksi lingkungan & GPU

In [ ]:
import os, sys, subprocess

IS_KAGGLE = os.path.exists('/kaggle')
IS_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')
ENV = 'Kaggle' if IS_KAGGLE else ('Colab' if IS_COLAB else 'lain')
print('Lingkungan   :', ENV)
print('Python       :', sys.version.split()[0])
try:
    out = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                         capture_output=True, text=True)
    print('GPU          :', out.stdout.strip() or '(tidak terdeteksi)')
except FileNotFoundError:
    print('GPU          : tidak ada (CPU-only) — aktifkan akselerator GPU!')

## 2. Siapkan kode `ocrpipe`

Pilih **salah satu**:
- **A.** Kalau sudah upload folder `ocrpipe` sebagai Kaggle Dataset, set `OCRPIPE_SRC` ke path-nya.
- **B.** Upload arsip `ocr-skripsi.zip` lewat sel upup di bawah.
- **C.** (Colab) mount Google Drive.

In [ ]:
from pathlib import Path
import shutil, zipfile

WORK = Path('/kaggle/working' if IS_KAGGLE else ('/content' if IS_COLAB else '.')).resolve()
os.chdir(WORK)

# --- Opsi A: dari Kaggle Dataset (ubah sesuai nama dataset-mu) ---
OCRPIPE_SRC = ''  # mis. '/kaggle/input/ocr-skripsi'

if OCRPIPE_SRC and (Path(OCRPIPE_SRC) / 'ocrpipe').exists():
    if (WORK / 'ocrpipe').exists():
        shutil.rmtree(WORK / 'ocrpipe')
    shutil.copytree(Path(OCRPIPE_SRC) / 'ocrpipe', WORK / 'ocrpipe')
    for f in ['run_ocr.py','cek_pdf.py']:
        if (Path(OCRPIPE_SRC)/f).exists():
            shutil.copy(Path(OCRPIPE_SRC)/f, WORK/f)
    print('ocrpipe disalin dari dataset ->', WORK)
else:
    print('OCRPIPE_SRC belum di-set / tidak ketemu. Pakai sel upload di bawah (Opsi B).')

In [ ]:
# --- Opsi B: upload arsip zip berisi folder ocrpipe/ (jalankan HANYA jika perlu) ---
# Colab:
if IS_COLAB:
    from google.colab import files
    up = files.upload()  # pilih ocr-skripsi.zip
    for name in up:
        if name.endswith('.zip'):
            with zipfile.ZipFile(name) as z:
                z.extractall(WORK)
            print('diekstrak:', name)
# Kaggle: upload lewat panel 'Add Data' > Upload, lalu set OCRPIPE_SRC di sel sebelumnya.
else:
    print('Di Kaggle, upload folder sebagai Dataset lalu pakai Opsi A.')

In [ ]:
assert (WORK / 'ocrpipe').exists(), 'Folder ocrpipe/ belum ada di working dir. Lihat Opsi A/B.'
sys.path.insert(0, str(WORK))
import ocrpipe
print('ocrpipe v' + ocrpipe.__version__, 'siap')

## 3. Pasang dependensi

Jalankan sel inti, lalu **satu** sel engine sesuai kebutuhan. Docling adalah default yang disarankan (lisensi MIT, tabel rapi).

In [ ]:
# inti (wajib)
!pip -q install pymupdf pdfplumber pypdf opencv-python-headless numpy pandas openpyxl pyyaml rapidfuzz

In [ ]:
# Tesseract + bahasa Indonesia (ringan; berguna juga sebagai mesin OCR Docling)
!apt-get -qq update && apt-get -qq install -y tesseract-ocr tesseract-ocr-ind > /dev/null
!pip -q install pytesseract
!tesseract --list-langs 2>/dev/null | tr '\n' ' '; print()

In [ ]:
# ENGINE A — Docling (disarankan)
!pip -q install docling

In [ ]:
# ENGINE B — PaddleOCR PP-StructureV3 (jalankan HANYA jika mau pakai paddle)
# GPU (CUDA 12.6):
# !pip -q install "paddleocr[doc-parser]"
# !pip -q install paddlepaddle-gpu==3.2.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
print('lewati sel ini kecuali butuh paddle')

In [ ]:
# ENGINE C — Marker (butuh GPU; perhatikan lisensi bobot)
# !pip -q install marker-pdf
print('lewati sel ini kecuali butuh marker')

In [ ]:
# cek engine yang terdeteksi
!python run_ocr.py engines

## 4. Siapkan PDF masukan

Taruh PDF di folder `data_pdf/`. Di Kaggle, upload sebagai Dataset lalu salin; di Colab pakai sel upload.

In [ ]:
DATA = WORK / 'data_pdf'
DATA.mkdir(exist_ok=True)

# Colab: upload PDF
if IS_COLAB:
    from google.colab import files
    up = files.upload()
    for name in up:
        if name.lower().endswith('.pdf'):
            shutil.move(name, DATA / name)

# Kaggle: salin dari dataset input (ubah path sesuai dataset PDF-mu)
PDF_DATASET = ''  # mis. '/kaggle/input/dokumen-akademik'
if PDF_DATASET and Path(PDF_DATASET).exists():
    for p in Path(PDF_DATASET).rglob('*.pdf'):
        shutil.copy(p, DATA / p.name)

pdfs = sorted(DATA.glob('*.pdf'))
print(f'{len(pdfs)} PDF di {DATA}:')
for p in pdfs: print('  -', p.name)

In [ ]:
# survei dulu: berapa yang perlu OCR?
!python cek_pdf.py data_pdf

## 5. Jalankan ekstraksi

Ganti `--engine` sesuai yang dipasang. GPU dipakai otomatis bila tersedia.

In [ ]:
!python run_ocr.py ingest data_pdf -o keluaran --engine docling

In [ ]:
# intip ringkasan QA
from pathlib import Path
qa = Path('keluaran/qa/laporan-qa.md')
if qa.exists():
    print(qa.read_text(encoding='utf-8')[:2500])

## 6. Kemas & unduh hasil

Semua keluaran (markdown, jsonl, tabel, qa, manifest) dizip menjadi satu berkas.

In [ ]:
import shutil
arsip = shutil.make_archive(str(WORK / 'hasil-ocrpipe'), 'zip', root_dir=str(WORK / 'keluaran'))
print('arsip dibuat:', arsip, f'({Path(arsip).stat().st_size/1e6:.1f} MB)')

if IS_COLAB:
    from google.colab import files
    files.download(arsip)
else:
    print('Kaggle: berkas ada di /kaggle/working/ — unduh lewat panel Output di kanan,')
    print('atau commit notebook agar hasil tersimpan sebagai Output.')

## 7. Langkah selanjutnya

- `keluaran/jsonl/semua-chunk.jsonl` siap di-ingest ke Qdrant (lihat `keluaran/contoh-ingest-qdrant.py`).
- Tinjau `keluaran/qa/laporan-qa.md` untuk halaman yang ditandai perlu dikoreksi manual.
- Untuk mengukur akurasi (CER/WER) demi Bab 4, jalankan `python run_ocr.py sample ...` lalu transkripsi manual dan `python run_ocr.py eval ...`.
- `keluaran/manifest.json` memuat versi semua pustaka + konfigurasi efektif — lampirkan sebagai bukti prosedur di skripsi.